# WSD 2 — Eksperimen Penyempurnaan Simplified Lesk

Melanjutkan WSD 1 (Lesk dasar, akurasi 61,1%). Dua ide penyempurnaan diuji di
ground truth yang sama (646 token), lalu diuji signifikannya dengan McNemar:

1. bobot **IDF** — kata konteks yang jarang muncul di gloss lain dianggap lebih penting
2. **stopword dibuang** dari konteks

Catatan: `Dataset Biologi.xlsx` (esai siswa) tidak punya label sense, jadi hanya
dipakai untuk demo penerapan di bagian akhir; pengukuran akurasi tetap di ground truth.

In [ ]:
# di Colab jalankan sekali: !pip install PySastrawi -q
import pandas as pd
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

df = None
for enc in ("utf-8-sig", "cp1252", "latin-1"):
    try:
        df = pd.read_csv("groundtruth_selesai.csv", sep=None, engine="python", encoding=enc)
        print("file terbaca dengan encoding :", enc)
        break
    except UnicodeDecodeError:
        pass

df["Sense_Benar"] = pd.to_numeric(df["Sense_Benar"], errors="coerce")
df = df.dropna(subset=["Sense_Benar"]).reset_index(drop=True)

print("jumlah token yang dievaluasi :", len(df))
print("jumlah kata unik             :", df["kata_token"].nunique())
print()
print(df[["kata_token", "Sense_Benar"]].head(5).to_string(index=False))

In [ ]:
# kamus mini dari ground truth: kata -> daftar gloss (arti) KBBI
def baca_gloss(teks):
    """pecah '1) ... | 2) ...' menjadi list gloss."""
    bagian = re.split(r"\s*\|\s*(?=\d+\)\s)", str(teks))
    return [re.sub(r"^\d+\)\s*", "", b).strip() for b in bagian if b.strip()]

KAMUS_KBBI = {}
for _, baris in df.iterrows():
    KAMUS_KBBI.setdefault(baris["kata_token"], baca_gloss(baris["Daftar_Gloss"]))

jumlah_ambigu = sum(1 for v in KAMUS_KBBI.values() if len(v) > 1)
print("kata unik di kamus mini :", len(KAMUS_KBBI))
print("kata ambigu (>1 sense)  :", jumlah_ambigu)

def lesk(kalimat, kata_target):
    """versi dasar (WSD 1): konteks unik, tanpa stemming. Return nomor sense."""
    if kata_target not in KAMUS_KBBI:
        return None
    konteks = list(dict.fromkeys(kalimat.split()))
    skor = [sum(1 for k in konteks if k != kata_target and k in g.split())
            for g in KAMUS_KBBI[kata_target]]
    return skor.index(max(skor)) + 1

benar = sum(lesk(b["Kunci_Jawaban"], b["kata_token"]) == b["Sense_Benar"]
            for _, b in df.iterrows())
print()
print(f"akurasi versi dasar : {benar}/{len(df)} = {benar/len(df)*100:.1f}%")

In [ ]:
# IDE 1: bobot IDF pada kata konteks
import math

semua_gloss = [g for daftar in KAMUS_KBBI.values() for g in daftar]
N = len(semua_gloss)
DF = {}
for g in semua_gloss:
    for w in set(g.split()):
        DF[w] = DF.get(w, 0) + 1
IDF = {w: math.log(N / d) for w, d in DF.items()}

def lesk_idf(kalimat, kata_target):
    """sama seperti lesk(), tapi overlap dibobot IDF."""
    konteks = list(dict.fromkeys(kalimat.split()))
    skor = []
    for gloss in KAMUS_KBBI[kata_target]:
        kata_gloss = set(gloss.split())
        nilai = sum(IDF.get(k, 0.0) for k in konteks
                    if k != kata_target and k in kata_gloss)
        skor.append(nilai)
    return skor.index(max(skor)) + 1

benar_idf = sum(lesk_idf(b["Kunci_Jawaban"], b["kata_token"]) == b["Sense_Benar"]
                for _, b in df.iterrows())
print(f"akurasi lesk + IDF : {benar_idf}/{len(df)} = {benar_idf/len(df)*100:.1f}%")
print()
print(">> TERNYATA TURUN dibanding versi tanpa IDF (395/646 = 61.1%)")
print(">> kata langka di gloss kecil kita belum tentu penting -> IDF TIDAK dipakai.")

In [ ]:
# IDE 2: buang stopword dari konteks
# kata seperti "yang, adalah, dengan" suka kebetulan ada di gloss -> bikin salah pilih
STOP = set(StopWordRemoverFactory().get_stop_words())
print("jumlah stopword Sastrawi :", len(STOP))
print("contoh stopword :", sorted(STOP)[:8])

def lesk_stopword(kalimat, kata_target):
    """lesk FINAL: konteks unik + stopword dibuang. Return nomor sense."""
    if kata_target not in KAMUS_KBBI:
        return None
    konteks = [k for k in dict.fromkeys(kalimat.split()) if k not in STOP]
    skor = [sum(1 for k in konteks if k != kata_target and k in g.split())
            for g in KAMUS_KBBI[kata_target]]
    return skor.index(max(skor)) + 1

benar_sw = 0
prediksi_baru = []
for _, baris in df.iterrows():
    p = lesk_stopword(baris["Kunci_Jawaban"], baris["kata_token"])
    prediksi_baru.append(p)
    if p == baris["Sense_Benar"]:
        benar_sw += 1

print()
print(f"akurasi lesk + stopword : {benar_sw}/{len(df)} = {benar_sw/len(df)*100:.1f}%")

In [ ]:
# rekap semua versi yang diuji di ground truth yang sama
rekap = pd.DataFrame({
    "versi": ["notebook awal (stemming)", "tanpa stemming + dedup",
              "+ bobot IDF", "+ stopword di konteks (FINAL)"],
    "benar": [370, 395, benar_idf, benar_sw],
})
rekap["akurasi %"] = (rekap["benar"] / len(df) * 100).round(1)
print(rekap.to_string(index=False))

In [ ]:
# uji signifikan McNemar (exact): versi sebelum vs sesudah stopword dibuang
from math import comb

prediksi_lama = [lesk(b["Kunci_Jawaban"], b["kata_token"]) for _, b in df.iterrows()]
jawaban = df["Sense_Benar"].astype(int).tolist()

b = sum(1 for l, n_, j in zip(prediksi_lama, prediksi_baru, jawaban) if l == j and n_ != j)
c = sum(1 for l, n_, j in zip(prediksi_lama, prediksi_baru, jawaban) if l != j and n_ == j)
n = b + c
p_value = min(1.0, 2 * sum(comb(n, i) for i in range(0, min(b, c) + 1)) / 2 ** n)

print(f"McNemar (lama vs FINAL): jadi benar {c} kata, jadi salah {b} kata, p = {p_value:.4f}")
print("p < 0.05 -> peningkatannya SIGNIFIKAN, bukan kebetulan")

naik = df[[l != j and n_ == j for l, n_, j in zip(prediksi_lama, prediksi_baru, jawaban)]]
print()
print("contoh kata yang JADI BENAR setelah stopword dibuang:")
print(naik[["kata_token", "Sense_Benar"]].head(8).to_string(index=False))

## Penerapan lesk FINAL ke esai biologi

Esai diambil dari `Dataset Biologi.xlsx` (bab Transpor Membran, dipakai juga di
bagian AES). Kamus mini dibuat dari kata-kata topik yang sama, jadi kata ambigu
seperti **sel, air, masuk, keluar, bagian, transparan** bisa langsung
didisambiguasi tanpa memanggil API KBBI lagi.

In [ ]:
# WSD semua kata ambigu di tiap jawaban siswa
NAMA_XLSX = "Dataset Biologi.xlsx"
data = pd.read_excel(NAMA_XLSX)
print("jumlah baris esai :", len(data))
print("jumlah siswa      :", data["Nama"].nunique())
print("jumlah soal       :", data["Kode_Soal"].nunique())

kata_ambigu = {k: v for k, v in KAMUS_KBBI.items() if len(v) > 1}
print("kata ambigu di kamus mini :", len(kata_ambigu))

def bersihkan(teks):
    """ambil kata-katanya saja (tanpa tanda baca & angka)."""
    return re.findall(r"[a-z]+", str(teks).lower())

hasil = []
for _, baris in data.iterrows():
    kata2 = bersihkan(baris["Jawaban Siswa"])
    for kata in sorted(set(kata2)):
        if kata in kata_ambigu and kata not in STOP:
            sense = lesk_stopword(" ".join(kata2), kata)
            hasil.append({
                "Kode_Soal": baris["Kode_Soal"],
                "kata_ambigu": kata,
                "sense_terpilih": sense,
                "gloss_terpilih": kata_ambigu[kata][sense - 1],
            })

hasil_wsd = pd.DataFrame(hasil)
print()
print("total kata ambigu ditemukan di semua esai :", len(hasil_wsd))
print()
print("kata ambigu yang paling sering muncul di esai siswa:")
print(hasil_wsd["kata_ambigu"].value_counts().head(10).to_string())

In [ ]:
print("contoh 5 hasil WSD di esai siswa:")
print(hasil_wsd.head(5).to_string(index=False))

hasil_wsd.to_excel("hasil_wsd_esai.xlsx", index=False)
hasil_wsd.to_csv("hasil_wsd_esai.csv", index=False, sep=";")
print()
print("hasil disimpan ke : hasil_wsd_esai.xlsx  (dan hasil_wsd_esai.csv pemisah ';')")

In [ ]:
print("=== CONTOH SATU KASUS ===")
contoh = hasil_wsd.iloc[0]
baris_asli = data[data["Kode_Soal"] == contoh["Kode_Soal"]].iloc[0]
print("soal  :", str(baris_asli["Pertanyaan"])[:80], "...")
print("kata  :", contoh["kata_ambigu"], "-> sense", contoh["sense_terpilih"])
print("gloss :", contoh["gloss_terpilih"])

print()
print("cara pakai untuk AES:")
print("  1. sense terpilih = info makna kata kunci di jawaban siswa")
print("  2. gloss sense bisa dibandingkan dengan Kunci Guru (kemiripan teks)")
print("  3. fitur ini bisa digabung dengan skor kemiripan biasa")